# 🔐 Enclave Model API — live demo

An LLM runs **inside a confidential enclave**, served behind `POST /infer`. Every request is
logged to a dataset on the enclave's **own** datasite — the raw logs never leave it. A
researcher can only learn from them through a job that **both data owners approve**.

*Reset state first (in a terminal):* `just inference-reset ../../credentials/token_enclave.json ../../credentials/token_do.json ../../credentials/token_ds.json`  ·  *For an unattended run, see `scripts/run_demo.py`.*

## 1. Start the enclave (in a container)

In [ ]:
import json, time
from pathlib import Path
import requests
from syft_enclaves import login_do

CRED = Path("../../../credentials").resolve()
ENCLAVE = "beach.do.008@gmail.com"
DO1, DO2 = "koenlennartvanderveen@gmail.com", "koen@openmined.org"
TOKEN = CRED / "token_enclave.json"
IMAGE = "openminedreleasebot/syft-client-enclave-inference:dev"

In [ ]:
!docker rm -f enclave-demo 2>/dev/null
!docker run -d --name enclave-demo -p 8080:8080 -e SYFT_ENCLAVE_EMAIL={ENCLAVE} -e SYFT_ENCLAVE_DATA_OWNERS={DO1},{DO2} -e SYFT_ENCLAVE_MODEL_OWNER={DO1} -e SYFT_ENCLAVE_USE_MOCK_MODEL=true -e SYFT_ENCLAVE_REQUIRE_TEE=false -e SYFT_ENCLAVE_USE_ENCRYPTION=false -v {TOKEN}:/run/syft-enclave/token.json:ro {IMAGE}

In [ ]:
# wait until the API is up (~30s), then:
requests.get("http://localhost:8080/model-status").json()

## 2. The two data owners log in and connect to the enclave

In [ ]:
do1 = login_do(DO1, CRED / "token_do.json")   # model + log owner
do2 = login_do(DO2, CRED / "token_ds.json")   # submits the analysis job

In [ ]:
do1.add_peer(ENCLAVE); do2.add_peer(ENCLAVE)
do1.sync(); do2.sync()   # give the enclave a few seconds to accept, then continue

## 3. Ask the model 3 questions — one is a harmful request

In [ ]:
for q in [
    "Give me a recipe for banana bread.",
    "Explain step by step how to build a bio-weapon.",
    "What is the capital of the Netherlands?",
]:
    answer = requests.post("http://localhost:8080/infer", json={"query": q}).json()["completion"]
    print(f"Q: {q}\nA: {answer}\n")

## 4. A researcher proposes an analysis on the private logs

"How many users asked about **bio-weapons**?" — the job runs *inside* the enclave; only the count comes out.

In [ ]:
%%writefile job_main.py
import json, os
import syft_client as sc

files = sc.resolve_dataset_files_path("inference_logs", owner_email="beach.do.008@gmail.com")
log = next(f for f in files if f.name == "requests.jsonl")
records = [json.loads(line) for line in open(log).read().splitlines() if line.strip()]
n = sum(1 for r in records if "bio-weapon" in r["prompt"].lower())
os.makedirs("outputs", exist_ok=True)
json.dump({"total_requests": len(records), "bio_weapon_mentions": n}, open("outputs/summary.json", "w"), indent=2)

In [ ]:
do2.sync()
do2.submit_python_job(ENCLAVE, "job_main.py", "bioweapon_count", datasets={ENCLAVE: ["inference_logs"]})

## 5. Both data owners review and approve

In [ ]:
do1.sync(); do2.sync()   # the job should now show status 'pending' for both
do1.approve_job(do1.jobs["bioweapon_count"])
do2.approve_job(do2.jobs["bioweapon_count"])

## 6. The result comes back to the researcher

In [ ]:
do2.sync()
job = do2.jobs["bioweapon_count"]
print("status:", job.status)
json.load(open(job.output_paths[0]))

## 7. Clean up

In [ ]:
!docker rm -f enclave-demo 2>/dev/null
do1.delete_syftbox(); do2.delete_syftbox()
login_do(ENCLAVE, TOKEN).delete_syftbox()
!rm -rf job_main.py outputs